In [2]:
import os
import csv
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

###############################################################################
#                           CONFIGURATION                                     #
###############################################################################

CSV_PATH = "dataset_split.csv"   # Your CSV file listing 400 videos
TRAIN_SPLIT = "train"
VAL_SPLIT   = "val"
TEST_SPLIT  = "test"

# Expected shape for each video: (T, H, W, 3), e.g. (120, 256, 256, 3)
# If you want to reduce input size, you can downsample frames:
RESIZE_HEIGHT = 224
RESIZE_WIDTH  = 224

# Number of frames per video (if all your videos are exactly the same length).
# If they vary, you can either pad or truncate to a fixed T.
FRAMES_PER_VIDEO = 120

# Training hyperparameters
BATCH_SIZE    = 2
EPOCHS        = 10
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

###############################################################################
#                         DATASET: VideoFrameDataset                          #
###############################################################################
class VideoFrameDataset(Dataset):
    """
    Loads a .npy video file: shape (T, H, W, 3).
    Transforms each frame into (3, H, W) via standard TorchVision transforms,
    and returns a stack of frames of shape (T, 3, H, W).

    We'll let the model handle the 2D CNN -> LSTM conversion.
    """
    def __init__(self, csv_file, split="train"):
        super().__init__()
        self.samples = []
        
        # Read CSV, gather entries that match the requested split
        with open(csv_file, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    filepath = row["filepath"]
                    label    = row["label"]
                    self.samples.append((filepath, label))
        
        # Build label-to-index mapping
        unique_labels = sorted(set([s[1] for s in self.samples]))
        self.label_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
        
        # Convert labels in samples to label indices
        self.samples = [(fp, self.label_to_idx[lab]) for (fp, lab) in self.samples]
        
        # Image transforms for each frame
        # Typical ImageNet mean/std used in ResNet
        self.frame_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((RESIZE_HEIGHT, RESIZE_WIDTH)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        video_path, label_idx = self.samples[idx]
        
        # Load frames: shape (T, H, W, 3)
        frames = np.load(video_path)  # e.g. (120, 256, 256, 3)
        
        # If frames vary in length, you might need to pad/truncate to FRAMES_PER_VIDEO
        # e.g. frames = frames[:FRAMES_PER_VIDEO] or zero-pad
        if len(frames) > FRAMES_PER_VIDEO:
            frames = frames[:FRAMES_PER_VIDEO]
        elif len(frames) < FRAMES_PER_VIDEO:
            # Simple zero-padding approach:
            pad_count = FRAMES_PER_VIDEO - len(frames)
            pad_shape = (pad_count, ) + frames.shape[1:]
            pad_array = np.zeros(pad_shape, dtype=frames.dtype)
            frames = np.concatenate([frames, pad_array], axis=0)

        # Now frames.shape == (FRAMES_PER_VIDEO, H, W, 3)
        
        # Transform each frame -> (3, H, W)
        transformed_frames = []
        for frame in frames:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # If frames are BGR
            tensor_frame = self.frame_transform(frame_rgb)      # shape (3, 224, 224)
            transformed_frames.append(tensor_frame)
        
        # Stack into (T, 3, 224, 224)
        video_tensor = torch.stack(transformed_frames, dim=0)  # shape (FRAMES_PER_VIDEO, 3, 224, 224)
        
        return video_tensor, label_idx

###############################################################################
#                 MODEL: 2D CNN (ResNet) + LSTM Architecture                 #
###############################################################################
class CNNLSTMModel(nn.Module):
    """
    This model:
      - Uses a pretrained 2D CNN (ResNet) to extract feature vectors per frame.
      - Feeds the sequence of frame-level features into an LSTM.
      - Outputs a classification over the video label.
    """
    def __init__(self, hidden_dim=256, num_classes=20):
        super(CNNLSTMModel, self).__init__()
        
        # 1. Load a pretrained ResNet (2D CNN) and remove its final FC layer
        #    For ResNet18, final features = 512-dim
        resnet = models.resnet18(pretrained=True)
        # Remove the final FC layer; we only want features
        # Another approach: self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        # But let's do something a bit more explicit:
        resnet.fc = nn.Identity()  # so output is (batch, 512)
        self.cnn = resnet
        self.feature_dim = 512  # For ResNet18

        # 2. LSTM: input_dim=512 (from CNN), hidden_dim=whatever we choose
        self.lstm = nn.LSTM(input_size=self.feature_dim,
                            hidden_size=hidden_dim,
                            batch_first=True)
        
        # 3. Final classification layer
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        """
        x: shape (batch_size, T, 3, H, W)
        
        We'll flatten the batch+time dims to pass frames through the CNN.
        Then we'll reshape to (batch, T, feature_dim) and pass that to LSTM.
        """
        B, T, C, H, W = x.shape
        # Merge batch & time: (B*T, C, H, W)
        x_reshaped = x.view(B*T, C, H, W)
        
        # Pass all frames through CNN -> shape (B*T, 512)
        with torch.no_grad():
            # If we want to fine-tune the CNN, remove "with torch.no_grad()"
            cnn_feats = self.cnn(x_reshaped)  # shape: (B*T, 512)
        
        # Reshape to (B, T, 512)
        cnn_feats = cnn_feats.view(B, T, self.feature_dim)
        
        # Pass sequence of features through LSTM
        lstm_out, (h_n, c_n) = self.lstm(cnn_feats)  # lstm_out: (B, T, hidden_dim)
        
        # Take the last time step's output
        last_out = lstm_out[:, -1, :]  # shape (B, hidden_dim)
        
        # Classify
        logits = self.fc(last_out)     # shape (B, num_classes)
        
        return logits

###############################################################################
#                         TRAINING & EVALUATION LOOP                          #
###############################################################################
def train_cnn_lstm():
    # 1. Create datasets
    train_dataset = VideoFrameDataset(CSV_PATH, split=TRAIN_SPLIT)
    val_dataset   = VideoFrameDataset(CSV_PATH, split=VAL_SPLIT)
    test_dataset  = VideoFrameDataset(CSV_PATH, split=TEST_SPLIT)
    
    # 2. Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)
    
    # 3. Number of classes
    num_classes = len(train_dataset.label_to_idx)
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test: {len(test_dataset)}")
    print(f"Number of classes: {num_classes}")
    
    # 4. Instantiate model, loss, optimizer
    model = CNNLSTMModel(hidden_dim=256, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 5. Training loop
    for epoch in range(EPOCHS):
        # ---------- TRAIN ----------
        model.train()
        total_train_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for batch_idx, (videos, labels) in enumerate(train_loader):
            videos = videos.to(device)   # (B, T, 3, H, W)
            labels = labels.to(device)   # (B,)
            
            optimizer.zero_grad()
            outputs = model(videos)      # shape (B, num_classes)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            # Accuracy
            _, preds = torch.max(outputs, dim=1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_acc = correct_train / total_train if total_train > 0 else 0
        
        # ---------- VALIDATION ----------
        model.eval()
        total_val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for videos, labels in val_loader:
                videos = videos.to(device)
                labels = labels.to(device)
                
                outputs = model(videos)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                
                _, preds = torch.max(outputs, dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)
        
        avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_acc = correct_val / total_val if total_val > 0 else 0
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
    
    # 6. FINAL TEST
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0
    
    with torch.no_grad():
        for videos, labels in test_loader:
            videos = videos.to(device)
            labels = labels.to(device)
            
            outputs = model(videos)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            _, preds = torch.max(outputs, dim=1)
            correct_test += (preds == labels).sum().item()
            total_test += labels.size(0)
    
    avg_test_loss = test_loss / len(test_loader) if len(test_loader) > 0 else 0
    test_acc = correct_test / total_test if total_test > 0 else 0
    
    torch.save(model.state_dict(), "2d_cnn_lstm.pth")
    print("Model saved.")

    print(f"\nFinal Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_acc*100:.2f}%")

###############################################################################
#                                ENTRY POINT                                  #
###############################################################################
if __name__ == "__main__":
    train_cnn_lstm()


Train samples: 280, Val samples: 60, Test: 60
Number of classes: 20
Epoch [1/10] | Train Loss: 2.9994, Train Acc: 6.79% | Val Loss: 2.9479, Val Acc: 11.67%
Epoch [2/10] | Train Loss: 2.8557, Train Acc: 11.07% | Val Loss: 2.9290, Val Acc: 6.67%
Epoch [3/10] | Train Loss: 2.7255, Train Acc: 14.29% | Val Loss: 2.8556, Val Acc: 8.33%
Epoch [4/10] | Train Loss: 2.6280, Train Acc: 20.71% | Val Loss: 2.8404, Val Acc: 8.33%
Epoch [5/10] | Train Loss: 2.4986, Train Acc: 26.07% | Val Loss: 2.7075, Val Acc: 13.33%
Epoch [6/10] | Train Loss: 2.3947, Train Acc: 25.71% | Val Loss: 2.8906, Val Acc: 13.33%
Epoch [7/10] | Train Loss: 2.2951, Train Acc: 31.43% | Val Loss: 2.6504, Val Acc: 21.67%
Epoch [8/10] | Train Loss: 2.1223, Train Acc: 40.00% | Val Loss: 2.5594, Val Acc: 28.33%
Epoch [9/10] | Train Loss: 2.0398, Train Acc: 39.29% | Val Loss: 2.4822, Val Acc: 16.67%
Epoch [10/10] | Train Loss: 1.9638, Train Acc: 39.64% | Val Loss: 2.3648, Val Acc: 30.00%
Model saved.

Final Test Loss: 2.3385, Test A